# Notebook 1: Software-Delivery Pipeline

The paper's primary worked example: a multi-agent SDLC pipeline.

```
generator → reviewer → {test, requirements, security} → merge
```

This notebook demonstrates:
1. Building the pipeline graph
2. Running `analyze_pipeline()` — the flagship one-call API
3. Interpreting the report: feasibility, masking, bottlenecks
4. What-if analysis: what happens when the reviewer degrades?
5. Visualization: masking dashboard, autonomy buffer, risk ranking

In [ ]:
from minimal_oversight import analyze_pipeline
from minimal_oversight.models import Node, PipelineGraph, AggregationType, GovernancePolicy
from minimal_oversight.capacity import check_feasibility, compute_pipeline_capacity
from minimal_oversight.topology import detect_motifs, rank_nodes_by_risk
from minimal_oversight.intervention import explain_failure_surface, compute_autonomy_time
from minimal_oversight.allocation import solve_amo, recommend_governance_changes
from minimal_oversight.simulation import simulate_pipeline, generate_synthetic_traces, SimulationConfig
from minimal_oversight import _formulae as F

import numpy as np

## 1. Define the pipeline

Parameters from the paper (Section 3, Experiment 5 and Motif 3):
- `σ_skill = 0.55` at every node
- `c = 0.65` catch rate
- `K/N = 0.50` review capacity
- Product aggregation at the merge gate

The reviewer has **fan-out degree 3** — making it the highest-leverage node.

In [ ]:
# Build the SDLC pipeline
gen = Node("generator", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50)
rev = Node("reviewer", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50)
test = Node("test", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50)
req = Node("requirements", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50)
sec = Node("security", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50)
merge = Node(
    "merge", sigma_skill=0.55, catch_rate=0.65, review_capacity=0.50,
    aggregation=AggregationType.PRODUCT,
)

pipeline = PipelineGraph([gen, rev, test, req, sec, merge])
pipeline.add_edge("generator", "reviewer")
pipeline.add_edge("reviewer", "test")
pipeline.add_edge("reviewer", "requirements")
pipeline.add_edge("reviewer", "security")
pipeline.add_edge("test", "merge")
pipeline.add_edge("requirements", "merge")
pipeline.add_edge("security", "merge")

print(pipeline)
print(f"Depth: {pipeline.depth}")
print(f"Sources: {pipeline.sources()}")
print(f"Sinks: {pipeline.sinks()}")

## 2. One-call analysis

The flagship API: pass the graph and a quality target, get a full report.

In [ ]:
report = analyze_pipeline(pipeline, p_min=0.50)
print(report)

## 3. Dig into the details

### 3a. Feasibility

In [ ]:
f = report.feasibility
print(f"Feasible: {f.feasible}")
print(f"C_op = {f.c_op:.3f}")
print(f"p_min = {f.p_min:.3f}")
print(f"B_eff = {f.b_eff:.4f}")
print(f"H_crit = {f.h_crit:.1f} bits")
print(f"Bottleneck: {f.bottleneck_node}")

### 3b. Per-node capacity and masking

In [ ]:
print(f"{'Node':<15} {'C_op':>8} {'σ_raw':>8} {'σ_corr':>8} {'M*':>8}")
print("-" * 50)
for name in pipeline.topological_order():
    node = pipeline.get_node(name)
    cap = report.node_capacities[name]
    sr = node.sigma_raw or 0
    sc = node.sigma_corr or 0
    m = node.masking_index or 0
    print(f"{name:<15} {cap:>8.3f} {sr:>8.3f} {sc:>8.3f} {m:>8.2f}")

### 3c. Detected motifs

The pipeline should show fan-out at the reviewer, merge at the merge gate,
and diamond patterns through the parallel branches.

In [ ]:
for m in report.motifs:
    print(f"[{m.motif.value}] {m.nodes}")
    print(f"  → {m.risk_description}")
    print()

### 3d. Intervention schedule

Which nodes need the most frequent human review?

In [ ]:
print(f"{'Rank':<6} {'Node':<15} {'T*_auto':>10}")
print("-" * 35)
for s in report.intervention_schedule:
    print(f"{s.priority_rank:<6} {s.node_name:<15} {s.t_auto:>10.1f}")

### 3e. Recommendations

In [ ]:
for r in report.recommendations:
    print(f"{r.priority}. [{r.target_node or 'pipeline'}] {r.action}")
    print(f"   Why: {r.rationale}")
    print(f"   Impact: {r.expected_impact}")
    print()

## 4. What-if analysis

### 4a. What if p_min is too ambitious?

The paper shows that when `p_min > C_op`, no governance policy can help.

In [ ]:
# Rebuild pipeline (fresh nodes)
def make_sdlc(sigma_skill=0.55, catch_rate=0.65, review_cap=0.50):
    nodes = [
        Node("generator", sigma_skill=sigma_skill, catch_rate=catch_rate, review_capacity=review_cap),
        Node("reviewer", sigma_skill=sigma_skill, catch_rate=catch_rate, review_capacity=review_cap),
        Node("test", sigma_skill=sigma_skill, catch_rate=catch_rate, review_capacity=review_cap),
        Node("requirements", sigma_skill=sigma_skill, catch_rate=catch_rate, review_capacity=review_cap),
        Node("security", sigma_skill=sigma_skill, catch_rate=catch_rate, review_capacity=review_cap),
        Node("merge", sigma_skill=sigma_skill, catch_rate=catch_rate, review_capacity=review_cap,
             aggregation=AggregationType.PRODUCT),
    ]
    p = PipelineGraph(nodes)
    p.add_edge("generator", "reviewer")
    p.add_edge("reviewer", "test")
    p.add_edge("reviewer", "requirements")
    p.add_edge("reviewer", "security")
    p.add_edge("test", "merge")
    p.add_edge("requirements", "merge")
    p.add_edge("security", "merge")
    return p

# Try increasingly ambitious targets
for target in [0.50, 0.60, 0.70, 0.80, 0.90]:
    p = make_sdlc()
    f = check_feasibility(p, p_min=target)
    status = "FEASIBLE" if f.feasible else "INFEASIBLE"
    print(f"p_min={target:.2f}  C_op={f.c_op:.3f}  B_eff={f.b_eff:.4f}  → {status}")

### 4b. Where should the expensive model go?

Paper Demonstration 1: placing a SOTA model as corrector at the reviewer
beats placing it as executor at the generator.

The reviewer's fan-out degree of 3 amplifies its improvement to all downstream branches.

In [ ]:
# Baseline
baseline = make_sdlc()
r_base = analyze_pipeline(baseline, p_min=0.50)
c_base = r_base.feasibility.c_op

# SOTA as executor at generator (σ_skill improves)
p_exec = make_sdlc()
p_exec.get_node("generator").sigma_skill = 0.92
r_exec = analyze_pipeline(p_exec, p_min=0.50)
c_exec = r_exec.feasibility.c_op

# SOTA as corrector at reviewer (catch_rate improves)
p_corr = make_sdlc()
p_corr.get_node("reviewer").catch_rate = 0.92
r_corr = analyze_pipeline(p_corr, p_min=0.50)
c_corr = r_corr.feasibility.c_op

print(f"Baseline C_op:                  {c_base:.4f}")
print(f"SOTA executor at generator:     {c_exec:.4f}  (Δ = {c_exec - c_base:+.4f})")
print(f"SOTA corrector at reviewer:     {c_corr:.4f}  (Δ = {c_corr - c_base:+.4f})")
print(f"\nRatio: corrector/executor = {(c_corr - c_base)/(c_exec - c_base):.1f}×")

### 4c. What happens when the reviewer degrades?

The fan-out structure means reviewer failure cascades to all three branches.

In [ ]:
print(f"{'Rev σ_skill':<15} {'C_op':>8} {'B_eff':>8} {'T*_auto':>10} {'Status':>12}")
print("-" * 58)
for skill in [0.55, 0.45, 0.35, 0.25, 0.15]:
    p = make_sdlc()
    p.get_node("reviewer").sigma_skill = skill
    r = analyze_pipeline(p, p_min=0.50)
    t_auto = r.intervention_schedule[0].t_auto if r.intervention_schedule else 0
    status = "OK" if r.feasibility.feasible and r.feasibility.b_eff > 0.05 else (
        "AT RISK" if r.feasibility.feasible else "INFEASIBLE"
    )
    print(f"{skill:<15.2f} {r.feasibility.c_op:>8.3f} {r.feasibility.b_eff:>8.4f} {t_auto:>10.1f} {status:>12}")

## 5. Simulation: watch masking evolve

Run the mean-field ODE simulator to see σ_raw and σ_corr converge.

In [ ]:
sim_pipeline = make_sdlc()
result = simulate_pipeline(sim_pipeline, SimulationConfig(n_steps=300, seed=42))

print(f"Simulated {result.config.n_steps} steps across {len(result.node_names)} nodes.")
print(f"\nFinal values (step 299):")
print(f"{'Node':<15} {'σ_raw':>8} {'σ_corr':>8} {'M*':>8}")
print("-" * 42)
for j, name in enumerate(result.node_names):
    sr = result.sigma_raw_history[-1, j]
    sc = result.sigma_corr_history[-1, j]
    m = result.masking_history[-1, j]
    print(f"{name:<15} {sr:>8.3f} {sc:>8.3f} {m:>8.2f}")

## 6. Estimation from traces

In production, you'd feed real logs. Here we generate synthetic traces
and show how estimation recovers the framework quantities.

In [ ]:
trace_pipeline = make_sdlc()
traces = generate_synthetic_traces(trace_pipeline, n_items=1000, seed=42)
print(f"Generated {len(traces)} synthetic traces.")

# Re-analyze with traces
trace_report = analyze_pipeline(make_sdlc(), p_min=0.50, traces=traces)

print(f"\n{'Node':<15} {'σ_raw':>8} {'σ_corr':>8} {'M*':>8} {'ĉ':>8} {'n':>6}")
print("-" * 55)
for name, est in trace_report.node_estimates.items():
    c_hat = f"{est.catch_rate:.3f}" if est.catch_rate is not None else "  n/a"
    print(f"{name:<15} {est.sigma_raw:>8.3f} {est.sigma_corr:>8.3f} {est.masking_index:>8.2f} {c_hat:>8} {est.sample_size:>6}")

## 7. Visualization

Requires `pip install minimal-oversight[viz]`.

In [ ]:
try:
    from minimal_oversight.viz import plot_masking_dashboard, plot_autonomy_buffer, plot_pipeline_risk
    HAS_VIZ = True
except ImportError:
    HAS_VIZ = False
    print("Install viz extras: pip install minimal-oversight[viz]")

In [ ]:
if HAS_VIZ:
    # Masking dashboard from trace estimates
    names = list(trace_report.node_estimates.keys())
    sr = [trace_report.node_estimates[n].sigma_raw for n in names]
    sc = [trace_report.node_estimates[n].sigma_corr for n in names]
    fig = plot_masking_dashboard(names, sr, sc)
    fig.savefig("masking_dashboard.png", dpi=150, bbox_inches="tight")
    print("Saved masking_dashboard.png")

In [ ]:
if HAS_VIZ:
    # Autonomy buffer vs process entropy
    fig = plot_autonomy_buffer(
        c_op=report.feasibility.c_op,
        p_min=0.50,
        governance_gap=0.02,
    )
    fig.savefig("autonomy_buffer.png", dpi=150, bbox_inches="tight")
    print("Saved autonomy_buffer.png")

In [ ]:
if HAS_VIZ:
    # Risk ranking
    risks = rank_nodes_by_risk(pipeline)
    rnames = [r.name for r in risks if r.sota_score is not None]
    scores = [r.sota_score for r in risks if r.sota_score is not None]
    m_indices = [r.masking_index or 1.0 for r in risks if r.sota_score is not None]
    fig = plot_pipeline_risk(rnames, scores, m_indices)
    fig.savefig("risk_ranking.png", dpi=150, bbox_inches="tight")
    print("Saved risk_ranking.png")

## 8. Failure surface explanation

In [ ]:
print(report.failure_explanation)

## Key takeaways

1. **The reviewer is the highest-priority node** — fan-out degree 3 means its correction propagates to all downstream branches.
2. **Masking is present at every node** — M* > 1.3 means the dashboard overstates competence by 30%+.
3. **The merge gate is the bottleneck** — product aggregation compounds errors from all parents.
4. **Placing the SOTA model as reviewer corrector beats executor at generator** — by an order of magnitude in system improvement.
5. **The pipeline is feasible at p_min=0.50 but infeasible at p_min=0.80** — no governance policy can bridge that gap; the topology must change.